# NARCliM2 Workflow — Example

**Author / Developer:** Jabbar Khaledi  
*Data and Geospatial Analyst | Python Developer*

This notebook demonstrates the complete NARCliM2 workflow: data download/extraction, temporal averaging and ensemble uncertainty, spatial summaries, configurable storm processing, and map generation.

For full package documentation, see `README.md`.


## Package Overview

The workflow processes NARCliM climate data through three main dimensions:

1. **Temporal averaging** — average each grid cell through the selected time horizon for each GCM × RCM member.
2. **Ensemble uncertainty** — calculate pixel-wise minimum, mean, and maximum across available model-member temporal averages.
3. **Spatial summarisation** — summarise the ensemble rasters for user-supplied polygon or point datasets.

For polygons, the reported `min`, `mean`, and `max` are spatial means of the corresponding ensemble rasters. For points, the values are sampled from the intersecting raster cell.


## Key Features

- NCI THREDDS / OPeNDAP data access
- polygon and point study areas
- automatic NARCliM domain selection
- configurable variables, GCMs, RCMs, scenarios, and time windows
- temporal-average rasters
- ensemble min/mean/max rasters
- configurable `StormDays` derived from daily `sfcWindmax`
- retained original storm-source data for QA
- GeoPackage and CSV spatial summaries
- representative `time_horizon_year`
- ensemble maps with study-area boundary overlay


## Domain Selection

| Domain | Resolution | Coverage |
|:--|:--:|:--|
| NARCliM2-0-SEAus-04 | ~4 km | South-East Australia |
| AUS-18 | ~18 km | Australia |

Use `domain_key="auto"`, `"seaus_4km"`, or `"aus_18"`.


## Variables configured in the package

| Variable | Description | Branch | Frequency |
|:--|:--|:--|:--:|
| `prAdjust` | Daily bias-adjusted precipitation | `bias-adjusted-output` | `day` |
| `tasmaxAdjust` | Daily bias-adjusted maximum temperature | `bias-adjusted-output` | `day` |
| `tasminAdjust` | Daily bias-adjusted minimum temperature | `bias-adjusted-output` | `day` |
| `TXge35` | Annual days with Tmax ≥ 35°C | `bias-adjusted-output` | `yr` |
| `TNlt2` | Annual days with Tmin < 2°C | `bias-adjusted-output` | `yr` |
| `FFDIgt50` | Annual days with FFDI ≥ 50 | `DD` | `yr` |
| `R20mm` | Annual days with precipitation ≥ 20 mm | `DD` | `yr` |
| `R99p` | Annual precipitation from extremely wet days | `DD` | `yr` |
| `SPI12` | 12-month Standardised Precipitation Index | `DD` | `mon` |
| `StormDays` | Annual days above user-defined daily max-wind threshold | `DD` / `sfcWindmax` | source `day`, derived `yr` |

### Storm processing

`StormDays` reads daily `sfcWindmax`, optionally saves the original daily study-area subset, and derives an annual threshold-exceedance count.

Example:

```python
storm_threshold_kmh=89.0
save_storm_source=True
```

creates `StormDaysGT89`. Rare `StormDaysGT*` outputs retain three decimal places after temporal/ensemble averaging.


## Workflow architecture

```text
workflow.py
    ↓
Download/extract NARCliM data
    ↓
uncertainty.py
    ↓
Temporal mean for each model + pixel-wise ensemble Min / Mean / Max
    ↓
spatial_summary.py
    ↓
Summarise hazards for planning zones, land use, districts, LGAs,
catchments, subcatchments, sites, assets, or other user datasets
    ↓
ensemble_maps.py
    ↓
Create maps from Ensemble_Stats rasters
```


## 1. Download NARCliM data


In [ ]:
from pathlib import Path
from narclim_workflow import run_workflow

VECTOR_PATH = Path(r"C:\path\to\study_area.shp")
OUTPUT_ROOT = Path(r"C:\NARCliM_Outputs")

TIME_WINDOWS = {
    "baseline": (1985, 2014),
    # "short_term": (2021, 2040),
    "mid_term": (2041, 2060),
    "long_term": (2081, 2100),
}

manifest = run_workflow(
    vector_path=VECTOR_PATH,
    output_root=OUTPUT_ROOT,
    time_windows=TIME_WINDOWS,
    variables=["prAdjust", "tasmaxAdjust", "tasminAdjust", "TXge35", "R99p", "FFDIgt50", "SPI12", "StormDays",],
    gcms=["ACCESS-ESM1-5", "EC-Earth3-Veg", "MPI-ESM1-2-HR", "NorESM2-MM", "UKESM1-0-LL",],
    scenarios=["historical", "ssp245", "ssp370"],
    rcms=["NARCliM2-0-WRF412R3", "NARCliM2-0-WRF412R5"],
    domain_key="auto",
    id_field=None,
    overwrite=False,
    print_traceback=True,
    storm_threshold_kmh=89.0,
    save_storm_source=True,
)

print(manifest["status"].value_counts(dropna=False))

### Download logic

```text
Select study area
    ↓
Select domain
    ↓
Loop through variables, scenarios, models, and time windows
    ↓
Read NCI THREDDS catalogue
    ↓
Open via OPeNDAP
    ↓
Subset time
    ↓
Extract relevant climate cells
    ↓
Save local NetCDF subsets
```


## 2. Uncertainty analysis


```text
Each GCM × RCM output
        ↓
Temporal average within selected horizon
        ↓
One temporal-average raster per member
        ↓
Pixel-wise ensemble Min / Mean / Max
        ↓
Three ensemble rasters
        ↓
Pixel-centroid ensemble summary GeoPackages
```


In [ ]:
from narclim_workflow.uncertainty import run_uncertainty_workflow

uncertainty_manifest = run_uncertainty_workflow(
    input_root=OUTPUT_ROOT,
    boundary_path=VECTOR_PATH,
    variables=[
        "TXge35",
        "R20mm",
        "FFDIgt50",
        "SPI12",
        "StormDaysGT89",
    ],
    scenarios=["historical", "ssp126", "ssp245", "ssp370"],
    time_windows=["baseline", "short_term", "mid_term", "long_term"],
    gcms=[
        "ACCESS-ESM1-5",
        "EC-Earth3-Veg",
        "MPI-ESM1-2-HR",
        "NorESM2-MM",
        "UKESM1-0-LL",
    ],
    rcms=["NARCliM2-0-WRF412R3", "NARCliM2-0-WRF412R5"],
    feature_ids=None,
    boundary_id_field=None,
    output_folder_name="Climate_Indicies",
    overwrite=False,
    verbose=True,
)

print(uncertainty_manifest["status"].value_counts(dropna=False))


## 3. Spatial summary


```text
Read point or polygon dataset
    ↓
Discover ensemble rasters
    ↓
Polygon → zonal spatial mean for ensemble min / mean / max
Point   → sample intersecting raster cell
    ↓
Add representative time_horizon_year
    ↓
Save GeoPackage + CSV
```


In [ ]:
from narclim_workflow.spatial_summary import run_spatial_summary

FEATURE_PATH = Path(r"C:\path\to\planning_zones.shp")
CLIMATE_INDICES_ROOT = OUTPUT_ROOT / "Climate_Indicies"

outputs = run_spatial_summary(
    feature_path=FEATURE_PATH,
    climate_indices_root=CLIMATE_INDICES_ROOT,
    dataset_name="Planning_Zones",
    feature_id_field=None,
    all_touched=True,
    overwrite=True,
    verbose=True,
)

You can restrict the spatial summary to selected variables, scenarios, or horizons by passing `variables=`, `scenarios=`, and `time_horizons=`. Set `dataset_name=None` to derive the output name from the input dataset.


## 4. Plots and maps


In [ ]:
from narclim_workflow.ensemble_maps import plot_ensemble_maps

maps = plot_ensemble_maps(
    input_root=OUTPUT_ROOT,
    boundary_path=VECTOR_PATH,
    variables=None,
    scenarios=None,
    statistics=["min", "mean", "max"],
    cmap="viridis",
    dpi=300,
    overwrite=True,
    show=False,
    verbose=True,
)

print(f"Created/found {len(maps)} map(s).")

## Output notes

- Temporal averages are calculated first within each model member.
- Ensemble min/mean/max are then calculated across model-member temporal averages.
- Polygon spatial-summary `min`/`mean`/`max` fields are spatial means of the corresponding ensemble rasters.
- Most exported climate values use one decimal place.
- `StormDaysGT*` retains three decimals because rare-event frequencies can be much smaller than 0.1 days/year.
- Spatial-summary outputs include `time_horizon_year`.
- Maps are written to `Climate_Indicies/Plot and Maps/`.
